# Day 1: Python and a first quantum circuit

Material development: **Gage Plott**.

Watch a short example, run it, make the indicated small change, and discuss what
happens. No previous Python experience is assumed. This notebook is a complete
worked example: you do not need to invent code to follow along.

**Before starting:** a working course environment and the **Python (PHYS 3810 Guided)**
kernel must already be available. Installation on unfamiliar laptops is not
included in the opening segment. If setup takes class time, use the shorter
route: cells and variables, the X circuit, then saving and restarting. The H
circuit can wait for the next demonstration. The existing preflight notebook
helps diagnose an environment; it is not the Python lesson.

## 1. What is a cell?

A notebook alternates explanations with code cells. Click a code cell and press
**Shift+Enter** to run it and move to the next cell. Run from top to bottom.
The number beside a code cell records execution order; it is not a line number.
An asterisk means it is still running. Wait for it to finish before continuing.

The first cell checks the selected interpreter. Run it unchanged; its imports
and assertions are support code, which we will not ask you to write.


In [ ]:
%matplotlib inline
import sys
from importlib.metadata import version

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0])
print("Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)


## 2. Names, numbers, and text

`=` gives a name a value. `512` is a whole number. `1 / 2` is division and gives
`0.5`. Quotation marks make text (a **string**); they are not printed as part of
the message. `print(...)` asks Python to show a value. A line beginning with `#`
is an explanatory comment.

Run this cell. Then replace only the text inside `greeting`'s quotation marks
with your own short greeting and run the cell again. Keep `repetitions` at 512
for the sampling examples. Running the cell replaces these values; it does not
add them to their previous values.


In [ ]:
greeting = "Hello, quantum lab!"
repetitions = 512
half = 1 / 2
print(greeting)
print("Repetitions:", repetitions)
print("One half:", half)


## 3. A small list

Square brackets hold a list of values. Python starts positions at **zero**:
`outcomes[0]` is the first item and `outcomes[1]` is the second. Here `"0"` and
`"1"` are text labels for measurement outcomes, not the numbers zero and one.


In [ ]:
outcomes = ["0", "1"]
print("All labels:", outcomes)
print("First label:", outcomes[0])
print("Second label:", outcomes[1])


## 4. Calling a function

A function performs an operation when you write its name followed by
parentheses. Values inside the parentheses are its **arguments**. `len` counts
items; `round` rounds a number. Commas separate arguments. `print` is also a
function. We will use this same call syntax to construct circuits.


In [ ]:
print("Number of outcome labels:", len(outcomes))
print("One half rounded to two decimal places:", round(half, 2))


## 5. Import the tools we will use

`from ... import ...` makes a library tool available under its name. Run this
cell once after each kernel restart. The plotting and exact-check tools below
are supporting machinery; learning their internals is not needed to use the
circuit examples.

A **terminal** is different from a notebook code cell. Installation commands
such as `python -m pip ...` belong in the terminal described by the setup guide,
not in a Python cell. The code cells here contain Python, except the first
`%matplotlib inline` line, which tells Jupyter to show figures in the notebook.


In [ ]:
from math import isclose, sqrt

import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

sampler = StatevectorSampler(seed=2026)


## 6. Make a one-qubit X circuit

`QuantumCircuit(1)` creates a new one-qubit circuit, initially in state $|0\rangle$.
`x_circuit.x(0)` applies an X gate to qubit 0. The dot means "use this object's
operation"; this is another function call. X changes $|0\rangle$ into
$|1\rangle$. **Predict:** which label should every ideal measurement produce?

The cell always makes a fresh circuit before adding its gate. Running it twice
therefore does not append a second X. The assertion is a software correctness
check, not a task for you to complete.


In [ ]:
x_circuit = QuantumCircuit(1)
x_circuit.x(0)
x_probabilities = Statevector.from_instruction(x_circuit).probabilities()
assert isclose(float(x_probabilities[1]), 1.0, rel_tol=0, abs_tol=1e-12)

x_diagram = x_circuit.draw("mpl", scale=1.2)
display(x_diagram)
plt.close(x_diagram)


## 7. Repeat the measurement and read a histogram

A **shot** is one measurement sample from a fresh run of the circuit. We request
512 shots. `measure_all(inplace=False)` makes a measured **copy**; the original
circuit stays unmeasured. The sampler runs locally on this computer with seed
2026, which makes this demonstration repeatable. It is not a hardware job.

The last few lines turn the results into counts and a plot. Read a count as
"how many of the 512 samples gave this label." Run the cell unchanged and check
your X prediction against the displayed count.


In [ ]:
x_measured = x_circuit.measure_all(inplace=False)
x_result = sampler.run([x_measured], shots=repetitions).result()[0]
x_counts = x_result.data.meas.get_counts()
assert x_counts == {"1": repetitions}
assert x_circuit.num_clbits == 0
print("X counts:", x_counts)

x_histogram = plot_histogram(x_counts, title=f"X gate: {repetitions} local samples")
display(x_histogram)
plt.close(x_histogram)


## 8. Make a fresh H circuit

H is the **Hadamard** gate. On $|0\rangle$, it makes an equal superposition of
$|0\rangle$ and $|1\rangle$, with equal measurement probabilities. This is a
state with amplitudes, not two separate classical records.

**Predict:** should the next histogram have one occupied bar or two? The exact
probabilities are one half for each outcome. A finite sample need not divide
exactly in half.


In [ ]:
h_circuit = QuantumCircuit(1)
h_circuit.h(0)
h_probabilities = Statevector.from_instruction(h_circuit).probabilities()
assert isclose(float(h_probabilities[0]), half, rel_tol=0, abs_tol=1e-12)
assert isclose(float(h_probabilities[1]), half, rel_tol=0, abs_tol=1e-12)
print("Exact H probabilities:", h_probabilities)

h_diagram = h_circuit.draw("mpl", scale=1.2)
display(h_diagram)
plt.close(h_diagram)


Run the next cell unchanged. Compare the two bars with the exact 50/50 prediction.
The final assertion allows ordinary finite-sample fluctuation; it does not
require the observed counts to be exactly equal.


In [ ]:
h_measured = h_circuit.measure_all(inplace=False)
h_result = sampler.run([h_measured], shots=repetitions).result()[0]
h_counts = h_result.data.meas.get_counts()
assert sum(h_counts.values()) == repetitions
assert set(h_counts) <= {"0", "1"}
assert abs(h_counts.get("1", 0) - repetitions * half) <= 6 * sqrt(repetitions / 4)
assert h_circuit.num_clbits == 0
print("H counts:", h_counts)

h_histogram = plot_histogram(h_counts, title=f"H gate: {repetitions} local samples; exact P(1)=1/2")
display(h_histogram)
plt.close(h_histogram)


## 9. Save, restart, and recover

Use **File > Save Notebook** (or Ctrl+S). A saved output is a record of an earlier
run; it does not prove that the current kernel has the variables needed later.
Use **Kernel > Restart Kernel and Run All Cells** to rebuild everything in order.
A restart clears Python's memory, while the notebook file keeps its code.

If a name is missing, check that its earlier cell ran. After editing an earlier
cell, run that cell itself with **Shift+Enter**, then run the later code cells
that use it. Editing text alone does not update Python's variables. For these short lessons, restarting and
running from the top is the reliable recovery route. Keep your working copy
separate from the original downloaded notebooks.

Discuss together: what changed between the X and H circuits? What does a bar
height measure? Why do repeated measurements of H not have to be exactly balanced?
These are local simulated examples, not a demonstration of quantum advantage.
